# Rebuild a chart directly from CSV

This notebook reads committed residue-level measurements rather than a pre-rendered chart. It reconstructs one conditional-expectation scaling slice and checks the plotted values.

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path
from statistics import median

import matplotlib.pyplot as plt

In [ ]:
repo_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'experiments' / 'results').is_dir()
)
grouped = defaultdict(list)
with (repo_root / 'experiments/results/discrepancy_dynamic.csv').open(newline='', encoding='utf-8') as stream:
    for row in csv.DictReader(stream):
        if row['group'] == 'scalable' and row['method'] == 'walk_ce' and int(row['rank']) == 4:
            grouped[int(row['K'])].append(float(row['ratio_to_dither']))

dimensions = sorted(grouped)
ratios = [median(grouped[k]) for k in dimensions]
print(list(zip(dimensions, [round(value, 4) for value in ratios])))

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(dimensions, ratios, color='#175c73', marker='o', linestyle='-')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='dither reference')
ax.set_xscale('log', base=2)
ax.set_xlabel('inner dimension K (rank 4)')
ax.set_ylabel('conditional-expectation error / dither')
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

assert dimensions == [64, 128, 256, 512, 1024]
assert all(value < 1.0 for value in ratios)

The source CSV contains every block and residue, so readers can replace the median, filter, or normalization instead of trusting one fixed rendering.